# Tutorial 3: cell-cell interaction, what are they saying to each other?

**ASI-FIMSA Workshop 2026, spatial omics, hands-on**

Tutorial 2 established *who stands next to whom*. But adjacency is not conversation. Two cells can
touch and ignore each other, and the interesting question for an immunologist is the next one: **is
there evidence of signalling across that contact, and between which populations?**

The standard answer is a **ligand-receptor** analysis. Take a curated list of known ligand-receptor
pairs and look for places where the ligand is expressed in one cell and its receptor in the cells
around it, more often than chance would give. We use
[**LIANA+**](https://www.nature.com/articles/s41556-024-01469-w), which is worth knowing about for
three reasons:

* **It is a consensus, not a method.** It runs five published scoring functions and aggregates their
  ranks, so a pair has to look good to several different statistics.
* **It separates spatial weighting from the score**, so you can run the *same* analysis with and
  without space and measure what space actually buys you.
* **Its interaction database encodes receptor complexes.** LFA-1 is `ITGAL_ITGB2`, not `ITGAL`. That
  is more faithful to the biology and stricter, and it changes which pairs are testable.

We work on the same 2,000 µm Crop as Tutorial 2, the same 16,006 cells, but with a different set of
genes, and Section 1 is about why that swap was necessary.

| Section | The question | What you get back |
| --- | --- | --- |
| **1** | Does this panel support the analysis at all? | 3,738 testable interactions |
| **3** | What does a consensus of five scores give you? | magnitude and specificity ranks |
| **5** | What does spatial weighting actually change? | a measured A/B, not a claim |
| **6** | Where in the tissue does one interaction happen? | per-cell scores and Moran's R |
| **7** | Which populations carry the pairs you care about? | a sender &rarr; receiver matrix |

Two things to take away: **your gene panel decides which questions you can ask**, and the top of a
ranked pair list should be read sceptically, because abundance drives it. Plus the immunology:
**CD47-SIRPA**, **CXCL12-CXCR4**, **ICAM1-ITGAL/ITGB2** and **TGFB1-TGFBR1/TGFBR2** localised to the
cell-type pairs that carry them, on real tissue.

Everything runs on a free Colab CPU session in two to three minutes of compute. No GPU.

---

## 0. Setup

### 0.1 Install

Colab already ships **numpy**, **pandas**, **matplotlib**, **scikit-learn**, **scipy** and
**seaborn**. We add **liana**, and **scanpy** for the AnnData ecosystem underneath it.

This takes about a minute.

In [ ]:
# ============================================================================
# Install. You do not need to read or understand this cell -- just run it.
# ============================================================================
import subprocess
import sys
import urllib.request

IN_COLAB = "google.colab" in sys.modules

REPO_RAW = "https://raw.githubusercontent.com/xiao233333/ASI-FIMSA-workshop-2026/main"
PACKAGES = ["scanpy", "seaborn", "liana"]

# Modules Colab may already have imported before your first cell runs. If pip
# replaces one of these underneath a running kernel, the in-memory copy is stale
# and the session has to be restarted -- so we check rather than hope.
WATCHED = ["numpy", "pandas", "scipy", "matplotlib"]
before = {}
for name in WATCHED:
    mod = sys.modules.get(name)
    if mod is not None:
        before[name] = getattr(mod, "__version__", None)

if IN_COLAB:
    args = list(PACKAGES)
    try:
        urllib.request.urlretrieve(f"{REPO_RAW}/constraints-colab.txt",
                                   "constraints-colab.txt")
        args = ["-c", "constraints-colab.txt"] + args
        print("using the Workshop pin set (constraints-colab.txt)")
    except Exception as exc:                       # noqa: BLE001
        print(f"could not fetch the pin set ({type(exc).__name__}); "
              "installing unpinned, which is usually fine")
    print("installing:", " ".join(PACKAGES), "... this takes about a minute")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)
    print("done")
else:
    print("Not running in Google Colab -- assuming liana, scanpy and seaborn")
    print("are already installed in this environment.")

# The restart banner. Only three packages in the whole Workshop can trigger it,
# and on this Tutorial's install line it should trigger for none of them.
from importlib.metadata import version as _v            # noqa: E402

changed = []
for name, old in before.items():
    try:
        new = _v(name)
    except Exception:                                   # noqa: BLE001
        continue
    if old is not None and new != old:
        changed.append(f"{name} {old} -> {new}")
if changed:
    print()
    print("=" * 68)
    print("RESTART THE RUNTIME before continuing:")
    print("   Runtime -> Restart session, then run this notebook from the top.")
    print("Replaced under a running kernel: " + "; ".join(changed))
    print("=" * 68)
else:
    print("no already-imported module was replaced -- no restart needed")

### 0.2 Imports

`import liana as li` pulls in scanpy, plotnine and numba on the way, so the first call is slow, five
to fifteen seconds, and every call after it is not. That is normal.

In [ ]:
import os
import time
import warnings
import zipfile
from importlib.metadata import version
from pathlib import Path

import anndata as ad
import liana as li
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 0
np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 100

for pkg in ("liana", "scanpy", "anndata", "numpy", "pandas"):
    print(f"{pkg:10s}: {version(pkg)}")

try:
    import spatialdata as sd
    HAVE_SPATIALDATA = True
    print(f"{'spatialdata':10s}: {version('spatialdata')}")
except Exception as exc:                           # noqa: BLE001
    HAVE_SPATIALDATA = False
    print(f"spatialdata: not available ({type(exc).__name__}) -- "
          "the H&E backdrop in Section 6 will be skipped")

### 0.3 Download the data

Two prepared files, both derived from the same Atera slide as Tutorial 2:

* **`atera_crop_lr.h5ad`** (5 MB), the analysis file: the same 16,006 Crop cells, the cell types you
  met in Tutorial 2, coordinates in micrometres, and **1,673 genes**, every connectomeDB2020 ligand
  or receptor the Atera run measures and detects in at least ten cells.
* **`atera_crop.zarr.zip`** (18 MB), the Tutorial 2 Crop, used here only for its H&E image so the
  interaction maps in Section 6 sit on tissue. If it fails to download, every figure still works
  without its backdrop.

In [ ]:
# ============================================================================
# Get atera_crop_lr.h5ad (required) and atera_crop.zarr.zip (optional backdrop)
# ============================================================================
HF_REPO = "xiao233333/asi-fimsa-workshop-2026"
LR_NAME = "atera_crop_lr.h5ad"
CROP_ZIP_NAME = "atera_crop.zarr.zip"
CROP_DIR = Path("atera_crop.zarr")
DATA_DIR = Path(os.environ.get("WORKSHOP_DATA_DIR", "."))
# Google Drive mirrors; the presenter sets these if Hugging Face is unreachable.
DRIVE_IDS = {
    LR_NAME: os.environ.get("WORKSHOP_LR_DRIVE_ID", ""),
    CROP_ZIP_NAME: os.environ.get("WORKSHOP_CROP_DRIVE_ID", ""),
}


def fetch(filename):
    """Local copy -> Hugging Face -> Google Drive. Returns a Path, or None."""
    for candidate in (DATA_DIR / filename, Path(filename)):
        if candidate.exists():
            print(f"{filename}: using local copy ({candidate})")
            return candidate
    try:
        from huggingface_hub import hf_hub_download
        p = Path(hf_hub_download(repo_id=HF_REPO, filename=filename,
                                 repo_type="dataset"))
        print(f"{filename}: downloaded from Hugging Face")
        return p
    except Exception as exc:                       # noqa: BLE001
        # Deliberately no traceback -- a wall of red text reads like something
        # you did wrong, and the Drive mirror below usually works.
        print(f"{filename}: Hugging Face not reachable ({type(exc).__name__})")
    if DRIVE_IDS.get(filename):
        try:
            import gdown
            gdown.download(id=DRIVE_IDS[filename], output=filename, quiet=True)
            if Path(filename).exists():
                print(f"{filename}: downloaded from the Google Drive mirror")
                return Path(filename)
        except Exception as exc:                   # noqa: BLE001
            print(f"{filename}: Drive mirror not reachable ({type(exc).__name__})")
    return None


lr_path = fetch(LR_NAME)
if lr_path is None:
    print()
    print("Could not fetch the analysis file. Please tell the presenter -- this is")
    print("our problem, not yours. If you have it already, put it next to this")
    print("notebook, or set WORKSHOP_DATA_DIR to the folder holding it, and re-run.")
else:
    print(f"  {lr_path.name}: {lr_path.stat().st_size / 1e6:.1f} MB")

crop_zip = fetch(CROP_ZIP_NAME)
if crop_zip is not None and not CROP_DIR.exists():
    # A .zarr.zip has to be unzipped before spatialdata.read_zarr() will open it.
    t0 = time.time()
    with zipfile.ZipFile(crop_zip) as zf:
        zf.extractall(CROP_DIR)
    print(f"  unzipped to {CROP_DIR}/ in {time.time() - t0:.1f} s")
HAVE_HE = CROP_DIR.exists() and HAVE_SPATIALDATA

---

## 1. The same cells, a different panel

This is the Crop you worked on in Tutorial 2: same tissue, same 16,006 cells, same cell type per
cell, same coordinates in micrometres. The only thing that changed is which genes came along.

In [ ]:
adata = ad.read_h5ad(lr_path)

# --------------------------------------------------------------------------
# One line of housekeeping that is not optional, and that costs an afternoon
# if nobody tells you. liana builds an internal table with
#     pd.DataFrame(...).reset_index().rename(columns={"index": "gene"})
# which assumes adata.var has an UNNAMED index: reset_index() names the new
# column after the index, so "index" only appears when there is no name. This
# file names its index "gene_name" (it is a real gene symbol index, which is
# good practice), so the rename silently does nothing and li.mt.bivariate later
# dies several frames deep with
#     MergeError: No common columns to perform merge on
# Clearing the two index names costs nothing and avoids it. Checked against
# liana 1.9.0; if a future release fixes this, these lines become harmless.
# --------------------------------------------------------------------------
adata.var.index.name = None
adata.obs.index.name = None

print(adata)
print()
print("coordinates (obsm['spatial']), in micrometres:")
print("  x:", adata.obsm["spatial"][:, 0].min().round(0), "to",
      adata.obsm["spatial"][:, 0].max().round(0))
print("  y:", adata.obsm["spatial"][:, 1].min().round(0), "to",
      adata.obsm["spatial"][:, 1].max().round(0))
print()
print("cell types:")
print(adata.obs["cell_type"].value_counts().to_string())

### 1.1 Normalise

The counts arrive raw, so we normalise per cell and `log1p`. LIANA+ works from `adata.X` and expects
it already transformed. It will not do this for you, and it will not warn you if you forget.

In [ ]:
# Cells with no counts at all across the LR panel can never score anything.
# normalize_total warns about them; this is what it is warning about.
n_zero = int((np.asarray(adata.X.sum(axis=1)).ravel() == 0).sum())
print(f"{n_zero} of {adata.n_obs:,} cells have zero counts across the whole LR "
      f"panel ({100 * n_zero / adata.n_obs:.2f}%) -- they can never score")

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

CELLTYPE_KEY = "cell_type"
SPATIAL_KEY = "spatial"
xy = adata.obsm[SPATIAL_KEY]
TYPES = list(adata.obs[CELLTYPE_KEY].cat.categories)
PALETTE = dict(zip(TYPES, adata.uns["cell_type_colors"]))
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes, {len(TYPES)} cell types")

### 1.2 Does this panel support the analysis at all?

Tutorial 2's Crop carries **69 genes**, chosen to name cell types: `EPCAM`, `KRT8`, `CD3D`, `CD68`,
`COL1A1` and so on. That panel is excellent at its job and useless at this one, because a
ligand-receptor test needs **both halves of a pair** to be measured.

The genes were never missing from the experiment. Atera is whole-transcriptome, 18,028 targets, so
the ligands and receptors were measured on this slide all along; they simply were not carried into
the file Tutorial 2 needed. The file we load here carries them instead.

The count depends on which curated database you ask, so we check two. A pair is testable only when
the ligand, the receptor, **and every subunit of any complex on either side** are measured.

In [ ]:
RESOURCE_NAME = "consensus"
resource = li.rs.select_resource(RESOURCE_NAME)
measured = set(adata.var_names)


def complex_is_measured(name):
    """A complex is 'A_B_C'; every subunit has to be on the panel."""
    return set(str(name).split("_")) <= measured


covered = resource[resource["ligand"].map(complex_is_measured)
                   & resource["receptor"].map(complex_is_measured)].drop_duplicates()

# connectomeDB2020 coverage on these same cells, recorded in the file when the
# panel was built. A second curated database, for contrast.
cdb = adata.uns["atera"]["lr_source"]["coverage"]["connectomeDB2020_lit"]

print(f"LIANA+ '{RESOURCE_NAME}' resource      : {len(resource):,} interactions")
print(f"  fully measurable on this panel   : {len(covered):,}")
print()
print("for contrast, connectomeDB2020 (literature-supported only):")
print(f"  pairs in the database            : {cdb['pairs_in_database']:,}")
print(f"  measurable on this 1,673-gene panel: {cdb['pairs_in_this_panel']:,}")
print(f"  measurable on Tutorial 2's 69 genes: {cdb['pairs_in_teaching_panel']:,}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 3.2))
labels = ["Tutorial 2 panel\n(69 genes)", "this panel\n(1,673 genes)"]
cdb_vals = [cdb["pairs_in_teaching_panel"], cdb["pairs_in_this_panel"]]
# The 69-gene number for the LIANA resource is not recorded in the file, so this
# bar shows only what we can actually compute here. No invented numbers.
lia_vals = [np.nan, len(covered)]

y = np.arange(2)
ax.barh(y - 0.2, cdb_vals, height=0.36, color="#bdbdbd",
        label="connectomeDB2020")
ax.barh(y + 0.2, [0 if np.isnan(v) else v for v in lia_vals], height=0.36,
        color="#1f6fb4", label="LIANA+ consensus (this Tutorial)")
for yi, v in zip(y - 0.2, cdb_vals):
    ax.text(v + 60, yi, f"{v:,}", va="center", fontsize=9)
ax.text(len(covered) + 60, 1.2, f"{len(covered):,}", va="center", fontsize=9)
ax.text(60, 0.2, "not computed here", va="center", fontsize=8, color="0.4",
        style="italic")
ax.set_yticks(y, labels)
ax.set_xlabel("testable interactions (every gene and complex subunit measured)")
ax.set_title("Two databases, the same 16,006 cells")
ax.legend(fontsize=8, loc="lower right")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

Two things to take from that.

**The panel decides what you may ask.** Whichever curated source you use, the 69-gene teaching panel
supports a handful of pairs and the 1,673-gene panel supports thousands. That gap is the content of
a panel decision, made months earlier by someone choosing which probes to order, arriving as a hard
limit on what you are allowed to conclude.

**The two databases are not interchangeable.** They disagree on how many interactions exist and on
which ones, because they were curated by different people applying different standards. A pair that
appears in one analysis and not another may be telling you about the *databases* rather than about
your tissue.

> Targeted spatial platforms (Xenium, CosMx, MERFISH) measure the genes on their panel and nothing
> else. A 300-gene immuno-oncology panel will support cell typing beautifully and ligand-receptor
> analysis barely, because curated interaction databases are built from the whole transcriptome and
> your panel is not. **Before you buy a panel, take the analysis you intend to run and intersect it
> with the probe list.** It takes ten minutes and it is the cheapest experiment you will ever do.

---

## 2. What a consensus method is doing

Most ligand-receptor tools run one statistic. LIANA+ runs five, each published on its own as *the*
way to score a pair, and then aggregates their rankings.

| Method | What it scores high |
| --- | --- |
| **CellPhoneDB** | ligand and receptor both highly expressed, versus a label-permutation null |
| **Connectome** | the pair is expressed *specifically* in this sender-receiver combination |
| **log2FC** | ligand and receptor are both up in their respective populations |
| **NATMI** (specificity edge) | the pair's expression is concentrated in this cell-type pair rather than spread |
| **SingleCellSignalR** (LRscore) | a regularised product of the two expression levels |

The aggregation is **RobustRankAggregate**, which asks how surprising each pair's *set* of ranks
would be if the five methods had ranked at random. You get two numbers per row: **`magnitude_rank`**
(is the interaction strongly expressed?) and **`specificity_rank`** (is it *particular* to this
sender &rarr; receiver pair, rather than something that happens everywhere?). Smaller is stronger,
and smaller is more specific.

Reading only one of the two is the commonest way to misuse this output. A pair can be enormous and
unremarkable, or faint and highly specific, and which you want depends on your question.

### 2.1 Receptor complexes, and how pairs get named

Most interesting receptors are not single proteins. LFA-1, the integrin a T cell uses to grip ICAM-1
during synapse formation, is a **heterodimer of ITGAL and ITGB2**; the TGF-beta receptor is a
**TGFBR1/TGFBR2 pair**. Some databases collapse these to a single representative gene, while LIANA+'s
consensus names the whole complex and requires **every subunit** to be measured and expressed.

That is stricter, and it means the pair you have in mind may be spelled differently here than in the
paper you read it in. Check before concluding a pair is missing.

In [ ]:
res_pairs = {(l, r) for l, r in zip(resource["ligand"], resource["receptor"])}

print("how the pairs an immunologist would name are spelled in this resource")
print("-" * 70)
for ligand, receptor in [("CD47", "SIRPA"), ("CXCL12", "CXCR4"),
                         ("ICAM1", "ITGAL"), ("TGFB1", "TGFBR2"),
                         ("VCAM1", "ITGA4"), ("IL16", "CD4"),
                         ("CSF1", "CSF1R"), ("B2M", "HLA-F")]:
    exact = (ligand, receptor) in res_pairs
    # the same ligand, but a receptor complex that CONTAINS the receptor gene
    complexes = sorted({r for (l, r) in res_pairs
                        if l == ligand and receptor in str(r).split("_")
                        and r != receptor})
    if exact:
        verdict = "single gene"
    elif complexes:
        verdict = "as a complex: " + ", ".join(complexes[:2])
    else:
        verdict = "ABSENT from this resource"
    print(f"  {ligand}_{receptor:14s} {verdict}")

`CD47_SIRPA` and `CXCL12_CXCR4` appear as plain gene pairs. `ICAM1`, `TGFB1` and `VCAM1` appear only
as complexes. **`CSF1_CSF1R` and `B2M_HLA-F` are not in this resource at all.**

That last one is worth sitting with. A pair that is absent from your database cannot be evaluated,
however real the biology. It is not refuted; it is never asked. When you read "we found no evidence
for X", the question that matters is whether X was in the search space at all.

---

## 3. The consensus, without any spatial information

Start with the non-spatial run. Every cell's position is ignored; only the cell-type labels and the
expression matrix are used. This is what CellPhoneDB-style analysis has always done, and it is the
baseline Section 5 measures spatial weighting against.

| Argument | Here | What it does |
| --- | --- | --- |
| `expr_prop` | `0.05` | every component must be detected in >= 5% of the relevant population |
| `min_cells` | `20` | populations smaller than this are dropped entirely |
| `n_perms` | `100` | permutations behind the CellPhoneDB p-values |

> `n_perms=100` is a compromise with the clock. The smallest p-value 100 permutations can resolve is
> 0.01, so anything reported as 0 means "below the resolution of this run", not "vanishingly small".
> For anything you intend to publish, set it to 1,000 or more.

This cell takes about twenty seconds.

In [ ]:
COMMON = dict(
    groupby=CELLTYPE_KEY,
    resource_name=RESOURCE_NAME,
    expr_prop=0.05,
    min_cells=20,
    use_raw=False,
    n_perms=100,
    seed=SEED,
    n_jobs=1,
    inplace=False,
    verbose=False,
)

t0 = time.time()
res_expr = li.mt.rank_aggregate(adata, **COMMON)
print(f"expression-only: {len(res_expr):,} rows in {time.time() - t0:.1f} s")
print("columns:", ", ".join(res_expr.columns))

In [ ]:
# Same-type rows (Tumour -> Tumour) dominate any count in a tissue where one
# population is 43% of the cells, and they are not what "cell-cell interaction"
# usually means. Drop them for every ranking in this Tutorial.
def cross_population(df):
    out = df.loc[df["source"] != df["target"]].copy()
    out["interaction"] = out["ligand_complex"] + " -> " + out["receptor_complex"]
    out["cell_pair"] = out["source"] + " -> " + out["target"]
    return out


cross_expr = cross_population(res_expr)
print(f"{len(cross_expr):,} cross-population rows "
      f"({100 * len(cross_expr) / len(res_expr):.0f}% of all rows)")
print()
print("top 15 by magnitude:")
print(cross_expr.sort_values(["magnitude_rank", "specificity_rank"])
      .head(15)[["source", "target", "ligand_complex", "receptor_complex",
                 "magnitude_rank", "specificity_rank"]].to_string(index=False))

### 3.1 Read the top of this list sceptically

A ranked list of interactions looks like a result, and the first thing to check is whether it is
reporting biology or reporting **statistical power**, which tracks abundance. Ask two questions of
the top 30 rows: which **molecules** keep appearing, and which **cell-type pairs** keep appearing.

In [ ]:
top = cross_expr.sort_values(["magnitude_rank", "specificity_rank"]).head(30)

fig, axes = plt.subplots(1, 2, figsize=(13.2, 4.4))
for ax, (col, title) in zip(axes, [("interaction", "which molecules"),
                                   ("cell_pair", "which cell-type pairs")]):
    share = top[col].value_counts()
    ax.barh(range(len(share)), share.values, color="#1f6fb4")
    ax.set_yticks(range(len(share)), share.index, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("rows in the top 30 by magnitude")
    ax.set_title(f"{title} own the top of the ranking")
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

n_inter = top["interaction"].nunique()
n_pairs = top["cell_pair"].nunique()
print(f"the top 30 rows use only {n_inter} distinct interactions "
      f"across {n_pairs} distinct cell-type pairs")
print()
print("the most repeated molecules, and how detected each half is:")
props = pd.Series(np.asarray((adata.X > 0).mean(axis=0)).ravel(), index=adata.var_names)
for interaction, n in top["interaction"].value_counts().head(4).items():
    ligand, receptor = interaction.split(" -> ")
    lp = np.min([props[g] for g in ligand.split("_")])
    rp = np.min([props[g] for g in receptor.split("_")])
    print(f"  {interaction:26s} {n} rows   detected in {lp:.0%} / {rp:.0%} of cells")

The right-hand panel is fairly flat: no single pair of populations owns the list. The left-hand
panel is not. A handful of interactions appear over and over with **different** senders and
receivers, and they are the same handful of very widely-detected genes. `GNAS` is detected in 74% of
the cells in this Crop, `ARF1` in 57%, `CDH1` in 53%.

That is the abundance problem in its non-spatial form. A gene detected in 74% of cells can pair with
almost anything, in almost any combination of populations, and the consensus keeps finding it
because all five underlying methods are functions of expression level. **The ranking reports
opportunity**, and running five methods does not fix it: they are five statistics over one matrix,
not five independent experiments.

---

## 4. The spatial scale, and a trap in the word "bandwidth"

A neighbourhood can be defined with a **hard radius**: two cells are neighbours within 30 µm, and
otherwise not. LIANA+ instead uses a **kernel**: every pair of cells gets a weight that decays
smoothly with distance, and `bandwidth` sets how fast.

The two are not the same parameter with different names, and the numbers are not comparable. With a
Gaussian kernel, `li.ut.spatial_neighbors` keeps every weight above `cutoff`, so the furthest cell
that still counts sits at about **2.15 &times; bandwidth** for the default `cutoff=0.1`. Hand LIANA+
a bandwidth of 30 µm expecting a 30 µm neighbourhood and you get one reaching 64 µm: twice the
tissue, four times the area, a different question.

In [ ]:
CUTOFF = 0.1
reach_factor = np.sqrt(2 * np.log(1 / CUTOFF))
print(f"with cutoff={CUTOFF}, effective reach = {reach_factor:.3f} x bandwidth")
print()
print("LIANA+ Gaussian bandwidths, and what they actually produce:")
rows = []
for bw in (10.0, 15.0, 20.0, 30.0):
    W = li.ut.spatial_neighbors(adata, bandwidth=bw, cutoff=CUTOFF,
                                max_neighbours=200, kernel="gaussian",
                                set_diag=False, spatial_key=SPATIAL_KEY,
                                inplace=False)
    # NOTE: W.nnz is NOT the edge count. spatial_neighbors zeroes below-cutoff
    # weights in place (`data * (data > cutoff)`) without pruning the sparse
    # structure, so nnz stays pinned at max_neighbours and reports a graph far
    # denser than the one that exists. Count the non-zero VALUES instead.
    deg = np.asarray((W > 0).sum(axis=1)).ravel()
    rows.append({"bandwidth (um)": bw, "reach (um)": round(bw * reach_factor, 1),
                 "median neighbours": int(np.median(deg)),
                 "mean": round(deg.mean(), 1),
                 "% isolated": round(100 * (deg == 0).mean(), 2)})
print(pd.DataFrame(rows).to_string(index=False))

**We use a 15 µm bandwidth**, which reaches about 32 µm, a couple of cell diameters. That is small
enough that a neighbour is plausibly a cell in contact, and large enough that most cells have
several.

### 4.1 A second bandwidth, for a different scale

A bandwidth appears in a second place, measuring something else entirely.
`li.ut.spatial_pair_proximity` scores how close two **cell types** sit, from a trimmed mean of the
distances between them. Those distances are population-scale, tens to hundreds of micrometres. Feed
it the 15 µm that was right for the cell graph and almost every cell-type pair scores ~0, which
would multiply nearly every cross-population score by nothing.

Two different bandwidths for two different scales is the consequence of one word being used for a
cell-to-cell decay and a population-to-population decay.

In [ ]:
LOCAL_BW = 15.0        # per-cell graph  (Section 6) -- reaches ~32 um
PROXIMITY_BW = 30.0    # cell-type proximity (Section 5) -- population scale

for bw in (LOCAL_BW, PROXIMITY_BW, 60.0):
    p = li.ut.spatial_pair_proximity(adata, groupby=CELLTYPE_KEY, kernel="gaussian",
                                     bandwidth=bw, spatial_key=SPATIAL_KEY,
                                     verbose=False)
    tag = {LOCAL_BW: "  <- too tight: annihilates the weighting",
           PROXIMITY_BW: "  <- used below"}.get(bw, "")
    print(f"bandwidth {bw:5.1f} um: proximity median {p['proximity'].median():.3f}, "
          f"{p['interacting'].mean():.0%} of pairs flagged interacting{tag}")

proximity = li.ut.spatial_pair_proximity(
    adata, groupby=CELLTYPE_KEY, kernel="gaussian", bandwidth=PROXIMITY_BW,
    spatial_key=SPATIAL_KEY, verbose=False,
)
print(f"\nbetween-population mean distances span "
      f"{proximity['mean_distance'].min():.0f} to {proximity['mean_distance'].max():.0f} um")

In [ ]:
mat = proximity.pivot(index="source", columns="target", values="proximity")
mat = mat.reindex(index=TYPES, columns=TYPES)

fig, ax = plt.subplots(figsize=(9.2, 7.4))
sns.heatmap(mat, cmap="mako", vmin=0, vmax=1, annot=True, fmt=".2f",
            annot_kws={"fontsize": 6}, linewidths=0.3, linecolor="white",
            cbar_kws={"label": "LIANA+ proximity weight"}, ax=ax)
ax.set_title(f"Directed cell-type proximity ({PROXIMITY_BW:.0f} um bandwidth)")
ax.set_xlabel("receiver / target")
ax.set_ylabel("sender / source")
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.show()

This heatmap is Tutorial 2's question asked with a different instrument: pure geometry, computed
before any ligand or receptor is looked at. Compare it with Tutorial 2's neighbourhood-enrichment
matrix, the tumour block, the endothelial &harr; perivascular pairing, the immune populations
sitting together. In Section 5 this matrix becomes the weight that multiplies every expression
score.

---

## 5. The same consensus, spatially weighted

Now run it again with `spatial_key` set. Everything else is identical, so the difference between the
two result tables is attributable to the spatial weighting and to nothing else.

Internally LIANA+ multiplies each interaction's scores by the proximity of its sender &rarr;
receiver pair before ranking, so a pair of populations that never meet has its scores multiplied
towards zero no matter how beautifully they co-express.

In [ ]:
t0 = time.time()
res_spatial = li.mt.rank_aggregate(
    adata,
    spatial_key=SPATIAL_KEY,
    spatial_kwargs={"kernel": "gaussian", "bandwidth": PROXIMITY_BW,
                    "trim_fraction": 0.1},
    **COMMON,
)
print(f"spatially weighted: {len(res_spatial):,} rows in {time.time() - t0:.1f} s")

cross_spatial = cross_population(res_spatial)
print()
print("top 15 by magnitude, spatially weighted:")
print(cross_spatial.sort_values(["magnitude_rank", "specificity_rank"])
      .head(15)[["source", "target", "ligand_complex", "receptor_complex",
                 "magnitude_rank", "specificity_rank"]].to_string(index=False))

### 5.1 What did spatial weighting change, and one trap in answering that

The obvious move is to join the two tables on the interaction and the cell-type pair and sort by how
far each row moved. Do exactly that and the answer will be wrong.

**RobustRankAggregate saturates at 1.0.** Any interaction not distinguishable from random across the
five methods gets `magnitude_rank = 1.0`: not "rank one", but the worst value the statistic has. A
large fraction of rows sit there, so sorting by movement puts rows that started pinned at the
ceiling on top, and their apparent promotion is an artefact of ties. Count the pinned rows, exclude
them, and rank what is left.

In [ ]:
ID_COLS = ["source", "target", "ligand_complex", "receptor_complex"]
ab = (cross_expr[ID_COLS + ["magnitude_rank"]]
      .merge(cross_spatial[ID_COLS + ["magnitude_rank"]], on=ID_COLS,
             suffixes=("_expression", "_spatial")))

pinned = ab["magnitude_rank_expression"] >= 1.0
print(f"cross-population rows compared      : {len(ab):,}")
print(f"pinned at the RRA ceiling (rank 1.0): {pinned.sum():,} "
      f"({100 * pinned.mean():.1f}%)  <- excluded below")

moved = ab.loc[~pinned & (ab["magnitude_rank_spatial"] < 1.0)].copy()
moved["log10_rank_change"] = (np.log10(moved["magnitude_rank_spatial"])
                              - np.log10(moved["magnitude_rank_expression"]))
print(f"rows unsaturated in BOTH runs       : {len(moved):,}")
print()
print("largest genuine promotions (negative = better rank once space is used):")
print(moved.sort_values("log10_rank_change").head(12).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.6, 5.0))

ax = axes[0]
ax.scatter(-np.log10(moved["magnitude_rank_expression"]),
           -np.log10(moved["magnitude_rank_spatial"]),
           s=6, alpha=0.25, lw=0, color="#1f6fb4", rasterized=True)
lim = [0, max(-np.log10(moved[["magnitude_rank_expression",
                               "magnitude_rank_spatial"]].to_numpy().min()), 1) + 0.4]
ax.plot(lim, lim, color="#C1272D", lw=1.4, ls="--", label="unchanged")
ax.set(xlim=lim, ylim=lim, xlabel="expression only  (-log10 magnitude rank)",
       ylabel="spatially weighted  (-log10 magnitude rank)",
       title=f"Every unsaturated cross-population row (n={len(moved):,})")
ax.legend(frameon=False, fontsize=9)
sns.despine(ax=ax)

ax = axes[1]
ax.hist(moved["log10_rank_change"], bins=60, color="#1f6fb4", alpha=0.85)
ax.axvline(0, color="#C1272D", lw=1.6, ls="--")
ax.set(xlabel="log10 rank change (negative = promoted by spatial weighting)",
       ylabel="rows", title="Almost everything moves, in both directions")
sns.despine(ax=ax)

plt.tight_layout()
plt.show()

pro = (moved["log10_rank_change"] < 0).mean()
dem = (moved["log10_rank_change"] > 0).mean()
print(f"promoted by spatial weighting: {pro:.1%} of unsaturated rows")
print(f"demoted  by spatial weighting: {dem:.1%}")
print(f"median |change|: {moved['log10_rank_change'].abs().median():.2f} log10 units")

The scatter is the summary, and it is not the picture people expect. Spatial weighting is **not a
filter** that removes a few implausible hits and leaves the rest alone. It re-orders almost
everything, because almost every score gets multiplied by a proximity that is not 1, and it promotes
and demotes in near-equal measure.

So "we used a spatially aware method" is not by itself a claim to have improved anything. What it
buys is specific: the rows near the top are pairs of populations that **demonstrably sit near each
other in this tissue**, which is a precondition for the biology that a non-spatial method assumes.
A constraint, not an accuracy.

What it does **not** buy you: the proximity weight is one number per *cell-type pair*, from a
trimmed mean over the whole Crop. It knows fibroblasts and T cells are often close; it does not know
whether *this* fibroblast is near *that* T cell. For that you need Section 6.

---

## 6. Where in the tissue? Per-cell scores

Everything so far has been per cell *type*. `li.mt.bivariate` computes a score at **every individual
cell**, from that cell's own ligand expression and its neighbours' receptor expression, weighted by
the spatial kernel. The output is a cells x interactions matrix you can put back on the slide.

It also returns **Moran's R** per interaction, the spatial autocorrelation of that local score. High
means the interaction is concentrated in coherent patches of tissue; near zero means it is scattered
everywhere and structured nowhere.

This is where `LOCAL_BW = 15 µm` is used, the cell-scale bandwidth from Section 4.

In [ ]:
li.ut.spatial_neighbors(adata, bandwidth=LOCAL_BW, cutoff=CUTOFF,
                        max_neighbours=200, kernel="gaussian", set_diag=False,
                        spatial_key=SPATIAL_KEY)
W = adata.obsp["spatial_connectivities"]
deg = np.asarray((W > 0).sum(axis=1)).ravel()
print(f"spatial graph at bandwidth {LOCAL_BW:.0f} um "
      f"(reach {LOCAL_BW * reach_factor:.0f} um):")
print(f"  real edges     : {int(deg.sum()):,}   ({deg.mean():.1f} per cell, "
      f"median {np.median(deg):.0f})")
print(f"  stored entries : {W.nnz:,}   <- what W.nnz reports, before pruning")
print(f"  isolated cells : {100 * (deg == 0).mean():.2f}%")

# Prune the explicit zeros. This is not cosmetic. Every below-cutoff weight is
# still SITTING IN the sparse matrix as a stored 0.0, so the local analysis
# below walks ~12x more entries than the graph actually has, for no change in
# the answer -- 0.0 contributes nothing to any weighted sum. One call fixes both
# the runtime and the misleading nnz.
W.eliminate_zeros()
print(f"  after eliminate_zeros(): {W.nnz:,} stored == real edges")

t0 = time.time()
lrdata = li.mt.bivariate(
    adata,
    resource_name=RESOURCE_NAME,
    local_name="cosine",
    global_name="morans",
    n_perms=100,
    seed=SEED,
    mask_negatives=False,
    add_categories=True,
    nz_prop=0.05,
    use_raw=False,
    verbose=False,
)
print(f"\nlocal analysis: {lrdata.n_obs:,} cells x {lrdata.n_vars:,} interactions "
      f"in {time.time() - t0:.1f} s")

local = lrdata.var.sort_values("morans", ascending=False)
print()
print(local.head(12)[["ligand", "receptor", "ligand_props", "receptor_props",
                      "morans"]].to_string())

The top of the Moran's R list is epithelial junction and adhesion biology: E-cadherin against its
several partners, tight-junction components, epithelial G-protein signalling. That is the abundance
story in its third costume. Those genes are expressed in the single largest compartment, which
occupies large contiguous blocks of this Crop, so their local scores are both high and spatially
coherent.

**Moran's R is not a significance test for signalling.** It says a score has spatial structure, and
a score can have beautiful spatial structure because the tissue has structure and the gene is
expressed in one part of it.

In [ ]:
# The H&E backdrop, the same helper Tutorial 2 uses: an RGB array plus an extent in
# micrometres taken from the element's own transformation, so nobody hard-codes
# a pixel size.
HE, HE_EXTENT = None, None
if HAVE_HE:
    import logging

    from spatialdata.transformations import get_transformation

    # ome-zarr logs its parsing at INFO, and liana's import chain lowers the
    # root logger level, at which point ome-zarr's chatter reaches the notebook
    # and buries the one line below that anyone needs.
    logging.getLogger("ome_zarr").setLevel(logging.WARNING)

    sdata = sd.read_zarr(CROP_DIR)
    el = sdata["he"]
    HE = np.asarray(el.transpose("y", "x", "c").data)
    M = get_transformation(el, "global").to_affine_matrix(
        input_axes=("x", "y"), output_axes=("x", "y"))
    h, w = HE.shape[:2]
    (x0, x1), (y0, y1) = (M @ np.array([[0, 0, 1], [w, h, 1]]).T)[:2]
    HE_EXTENT = (x0, x1, y1, y0)          # y flipped: images draw top-down
    print(f"H&E backdrop {HE.shape} covering {tuple(round(v) for v in HE_EXTENT)}")
else:
    print("no H&E available -- the maps below will be drawn on white")


def _backdrop(ax):
    """Desaturated H&E. Score colour against pink-and-purple tissue is
    unreadable; the backdrop is here for anatomical context, not its own sake."""
    if HE is None:
        ax.invert_yaxis()
        return
    grey = HE.mean(axis=2)
    ax.imshow(grey, extent=HE_EXTENT, cmap="gray", alpha=0.9,
              vmin=np.percentile(grey, 2), vmax=np.percentile(grey, 98))
    ax.set_xlim(HE_EXTENT[0], HE_EXTENT[1])
    ax.set_ylim(HE_EXTENT[2], HE_EXTENT[3])


def _dense(mat):
    return np.asarray(mat.todense()).ravel() if hasattr(mat, "todense") \
        else np.asarray(mat).ravel()


def complex_expression(name):
    """A complex is only as expressed as its least-expressed subunit."""
    parts = str(name).split("_")
    return np.min(np.column_stack([_dense(adata[:, g].X) for g in parts]), axis=1)


def plot_local(interaction, suptitle=None):
    ligand = str(lrdata.var.loc[interaction, "ligand"])
    receptor = str(lrdata.var.loc[interaction, "receptor"])
    pvals = _dense(lrdata[:, interaction].layers["pvals"])
    panels = [
        (complex_expression(ligand), f"ligand: {ligand}", "viridis"),
        (complex_expression(receptor), f"receptor: {receptor}", "viridis"),
        (_dense(lrdata[:, interaction].X), f"local score: {interaction}", "magma"),
        (-np.log10(np.clip(pvals, 1e-2, 1)), "local evidence (-log10 p, floor 0.01)",
         "magma"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(12.4, 10.4))
    for ax, (values, title, cmap) in zip(axes.ravel(), panels):
        _backdrop(ax)
        pts = ax.scatter(xy[:, 0], xy[:, 1], c=values, s=3.2, cmap=cmap, lw=0,
                         alpha=0.9, rasterized=True)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(title, fontsize=10)
        fig.colorbar(pts, ax=ax, fraction=0.045, pad=0.03)
    fig.suptitle(suptitle or "Expression and local spatial coordination",
                 y=0.99, fontsize=13)
    plt.tight_layout()
    plt.show()


TOP_LOCAL = local.index[0]
print("top interaction by Moran's R:", TOP_LOCAL)
plot_local(TOP_LOCAL, f"{TOP_LOCAL} -- the most spatially structured local score")

### 6.1 Now a pair chosen for its biology instead of its rank

The figure above shows the interaction the *statistic* liked best, which is a different thing from
the interaction you came to ask about. **`CXCL12-CXCR4`** is the chemokine axis that positions
lymphocytes throughout the tumour microenvironment, so an immunologist would want to see it whether
or not it climbed the list. It is measurable here, and it is *not* at the top of the Moran's R
ranking. Ask it directly.

In [ ]:
NAMED = "CXCL12^CXCR4"
if NAMED in lrdata.var_names:
    rank = list(local.index).index(NAMED) + 1
    print(f"{NAMED}: Moran's R {local.loc[NAMED, 'morans']:+.3f}, "
          f"ranked {rank} of {lrdata.n_vars} -- nowhere near the top")
    print(local.loc[[NAMED]][["ligand", "receptor", "ligand_props",
                              "receptor_props", "morans"]].to_string())
    plot_local(NAMED, f"{NAMED} -- chosen for its biology, not its rank")
else:
    print(f"{NAMED} did not survive nz_prop=0.05 in this run.")

Compare the two figures. The top-ranked interaction paints the epithelial blocks; this one paints
the interstitial stroma **between** them and avoids the tumour nests, which is the same
immune-exclusion geometry that Tutorial 2's niches drew. Two analyses of the same tissue, reached
from different directions: Tutorial 2 from which cell types are near each other, this one from which
transcripts are. When two independent analyses agree on where the action is, the structure is
probably real.

---

## 7. Ask the immunology directly

Section 3.1 argued that the ranked list reports opportunity, so do not read down it. Take the pairs
an immunologist would name unprompted, and ask each one where it acts.

`magnitude_rank` and `specificity_rank` are both shown on purpose. A pair can be top of one and
unremarkable on the other, and the disagreement is information.

In [ ]:
# Named with their LIANA+ consensus spelling -- Section 2.1 is where the
# complex names come from.
WATCHED = [
    ("CD47", "SIRPA", "the 'don't eat me' axis; an immuno-oncology drug target"),
    ("CXCL12", "CXCR4", "chemokine positioning of lymphocytes"),
    ("ICAM1", "ITGAL_ITGB2", "ICAM-1 to LFA-1: the immunological synapse's grip"),
    ("TGFB1", "TGFBR1_TGFBR2", "the dominant immunosuppressive axis in solid tumours"),
    ("IL16", "CD4", "CD4 ligation; antigen-presentation adjacent"),
    ("SPP1", "CD44", "osteopontin; tumour-associated macrophage biology"),
    ("LGALS1", "PTPRC", "galectin-1 to CD45; T-cell restraint"),
    ("VCAM1", "ITGA4_ITGB1", "VCAM-1 to VLA-4: adhesion and extravasation"),
    ("CD274", "PDCD1", "PD-L1 to PD-1: the checkpoint everyone asks about"),
]

rows = []
for ligand, receptor, _note in WATCHED:
    hit = cross_spatial[(cross_spatial["ligand_complex"] == ligand)
                        & (cross_spatial["receptor_complex"] == receptor)]
    if hit.empty:
        rows.append({"interaction": f"{ligand} -> {receptor}", "cell pairs": 0,
                     "strongest sender -> receiver": "filtered out "
                     "(too sparse, or absent from the resource)",
                     "magnitude": np.nan, "specificity": np.nan,
                     "runner-up": ""})
        continue
    hit = hit.sort_values(["magnitude_rank", "specificity_rank"])
    best = hit.iloc[0]
    rows.append({
        "interaction": f"{ligand} -> {receptor}",
        "cell pairs": len(hit),
        "strongest sender -> receiver": best["cell_pair"],
        "magnitude": best["magnitude_rank"],
        "specificity": best["specificity_rank"],
        "runner-up": hit.iloc[1]["cell_pair"] if len(hit) > 1 else "",
    })

watched = pd.DataFrame(rows)
print(watched.to_string(index=False, float_format=lambda v: f"{v:.2e}"))

In [ ]:
# For every watched pair that survived, which sender -> receiver combinations
# carry it? Heatmaps of -log10(magnitude_rank): brighter = stronger consensus.
present = [(l, r) for l, r, _ in WATCHED
           if not cross_spatial[(cross_spatial["ligand_complex"] == l)
                                & (cross_spatial["receptor_complex"] == r)].empty]
show = present[:4]

fig, axes = plt.subplots(2, 2, figsize=(14.5, 11.5))
for ax, (ligand, receptor) in zip(axes.ravel(), show):
    sub = cross_spatial[(cross_spatial["ligand_complex"] == ligand)
                        & (cross_spatial["receptor_complex"] == receptor)]
    mat = sub.pivot_table(index="source", columns="target",
                          values="magnitude_rank", aggfunc="min")
    mat = -np.log10(mat.reindex(index=TYPES, columns=TYPES))
    keep = mat.index[mat.notna().any(axis=1) | mat.notna().any(axis=0)]
    mat = mat.loc[keep, keep]
    sns.heatmap(mat, cmap="magma_r", ax=ax, square=True, linewidths=0.3,
                linecolor="white", cbar_kws={"shrink": 0.62,
                                             "label": "-log10 magnitude rank"})
    ax.set_title(f"{ligand} -> {receptor}", fontsize=11)
    ax.set_xlabel("receiver")
    ax.set_ylabel("sender")
    ax.tick_params(labelsize=7)
fig.suptitle("Where each named interaction acts (blank = below expr_prop, "
             "or populations too small)", y=1.0, fontsize=12)
plt.tight_layout()
plt.show()

### 7.1 Which populations carry them

For every watched pair that survived the filters, which sender &rarr; receiver combination carries
it? This is the table an immunologist actually wants, and it is a different question from "what is
at the top of the list".

In [ ]:
survived = watched[watched["cell pairs"] > 0]
tops = survived["strongest sender -> receiver"]
tcell_dc = tops.str.contains("T cell").__and__(tops.str.contains("Dendritic cell"))
plasma = tops.str.startswith("Plasma cell")

print(f"pairs that survived filtering              : {len(survived)} of {len(watched)}")
print(f"  topping out on T cell <-> Dendritic cell : {tcell_dc.sum()}")
print(f"  topping out on Plasma cell -> ...        : {plasma.sum()}")
print()
counts = adata.obs[CELLTYPE_KEY].value_counts()
print("population sizes behind those claims:")
for t in ["T cell", "Dendritic cell", "Macrophage", "Plasma cell", "Mast"]:
    print(f"  {t:16s} {counts[t]:>6,} cells")

**One axis dominates the immune half of this tumour: T cell &harr; dendritic cell.** It tops four of
the eight surviving pairs. `ICAM1-ITGAL/ITGB2` is ICAM-1 binding LFA-1, the adhesion step of the
immunological synapse, and `IL16-CD4` and `B2M-HLA-F` are antigen-presentation-adjacent. Finding
them on the same pair of populations is coherent: this is what T cells meeting antigen-presenting
cells looks like when all you can see is transcripts.

**And something else tops three more: `Plasma cell -> T cell`.** Before enjoying that, look at the
population sizes printed above. There are **91 plasma cells** in this Crop. A population that small
can reach a good `magnitude_rank` on a handful of cells in one corner of the tissue, and the
consensus cannot tell that apart from a real axis. Treat it as a hypothesis with an obvious next
test rather than as a finding.

The abundance argument applies to the T cell &harr; dendritic cell result too. T cells (1,590) and
dendritic cells (928) are the two largest immune populations here, so the method has more power on
them, and that is a live alternative explanation for why they keep appearing.

Note also what is **absent**. `CD274-PDCD1`, the checkpoint every clinician in the room will ask
about, is filtered out entirely: PD-L1 and PD-1 transcripts are far too sparse in this Crop to clear
`expr_prop=0.05`. **That is a statement about detection, not about the tumour.** And there is no
B cell population in this Crop at all, so every B-cell axis is unanswerable here no matter how many
genes we carried. **A missing cell type is a harder limit than a missing gene.**

---

## 8. Sensitivity: does the answer depend on the number you chose?

The bandwidth is a biological assumption wearing a numeric disguise. The way to report one is to
show what happens either side of it, and to fix the value *before* looking at which one gives the
nicest answer.

In [ ]:
FOCUS = [("Fibroblast", "T cell"),
         ("T cell", "Dendritic cell"),
         ("Endothelial", "Perivascular"),
         ("Tumour epithelial", "T cell")]

rows = []
for bw in (15.0, 30.0, 45.0, 60.0):
    p = li.ut.spatial_pair_proximity(adata, groupby=CELLTYPE_KEY, kernel="gaussian",
                                     bandwidth=bw, spatial_key=SPATIAL_KEY,
                                     verbose=False)
    for source, target in FOCUS:
        sel = p[(p["source"] == source) & (p["target"] == target)]
        if not sel.empty:
            rows.append({"bandwidth_um": bw, "cell_pair": f"{source} -> {target}",
                         "proximity": float(sel["proximity"].iloc[0])})
sens = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8.0, 4.6))
sns.lineplot(data=sens, x="bandwidth_um", y="proximity", hue="cell_pair",
             marker="o", linewidth=2.2, ax=ax)
ax.axvline(PROXIMITY_BW, color="0.5", ls="--", lw=1.2)
ax.text(PROXIMITY_BW, ax.get_ylim()[1], " chosen", va="top", fontsize=9, color="0.4")
ax.set(xlabel="Gaussian bandwidth (um)", ylabel="LIANA+ proximity",
       title="Neighbourhood-scale sensitivity")
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

print(sens.pivot(index="cell_pair", columns="bandwidth_um",
                 values="proximity").round(3).to_string())

Every curve rises: widen the neighbourhood and everything is closer to everything. What matters is
whether the **ordering** is stable, because the ordering is what the weighting acts on. Where two
curves cross, a conclusion that depends on which of those two pairs is "closer" is a conclusion that
depends on your bandwidth, and should be reported as such.

---

## 9. What this does not show

Every figure above is a statement about **transcript co-location**, and it is worth being precise
about the distance between that and "these cells are signalling".

* **mRNA is not protein.** A cell transcribing `CSF1R` may not display the receptor; a displayed
  receptor may not be engaged. Nothing here measures binding.
* **Neighbourhood is not contact.** Two cells 25 µm apart are neighbours under our kernel and are not
  touching. Every result moves if you change the bandwidth.
* **The pair list is somebody's literature review.** A genuine interaction that is not in the
  resource cannot be found, and a pair in the resource that does not occur in breast tissue can still
  come back significant.
* **The ranking follows abundance.** Immune signalling is transcriptionally quiet relative to matrix
  and housekeeping production, and will essentially never top this list.
* **Sparse counts, single cells.** Most cell-by-gene entries are zero, and a pair only scores where
  both halves happen to be detected. Absence of signal is weak evidence of absence of signalling.
* **One Crop, one specimen**, 2,000 µm of one tumour with no replicates.
* **A consensus of five methods is not five independent opinions.** All five read the same expression
  matrix and four are monotone in expression level, so agreement between them is much weaker evidence
  than agreement between two genuinely different experiments.
* **The spatial weighting in Section 5 is per cell type, not per cell.** It is one number for each
  ordered pair of populations. Only Section 6 works at cell resolution.
* **`magnitude_rank = 1.0` is a ceiling, not a measurement.** Roughly a fifth of cross-population
  rows sit there, so any analysis that sorts by rank change has to exclude them first.

None of this makes the analysis useless. It makes it a **hypothesis generator with coordinates**,
which is a great deal more than a bulk experiment can give you and a great deal less than proof.

---

## Further reading

**The method used here.** [Dimitrov *et al.*, "LIANA+ provides an all-in-one framework for
cell-cell communication inference", *Nature Cell Biology* 26, 1613-1622
(2024)](https://www.nature.com/articles/s41556-024-01469-w), the framework, the consensus scheme and
the spatial extensions this Tutorial uses. [Documentation](https://liana-py.readthedocs.io/).

**The rank aggregation.** [Kolde *et al.*, "Robust rank aggregation for gene list integration and
meta-analysis", *Bioinformatics* 28, 573-580 (2012)](https://doi.org/10.1093/bioinformatics/btr709),
where `magnitude_rank` comes from and why it saturates at 1.

**The pair list behind this panel.** [Hou *et al.*, "Predicting cell-to-cell communication networks
using NATMI", *Nature Communications* 11, 5011
(2020)](https://www.nature.com/articles/s41467-020-18873-z), connectomeDB2020, which decided which
1,673 genes the analysis file carries.

**Other tools for the same question.**

* [**CellPhoneDB**](https://www.nature.com/articles/s41596-020-0292-x): the most widely used
  ligand-receptor method, and the one your reviewers will know. Non-spatial by default, permuting
  cluster labels rather than using positions. `squidpy.gr.ligrec` runs the same statistic.
* [**CellChat**](https://www.nature.com/articles/s41467-021-21246-8): R, and stronger at *organising*
  results, grouping pairs into signalling pathways so you read "TGFb signalling" instead of eleven
  separate rows.
* [**NicheNet**](https://www.nature.com/articles/s41592-019-0667-5): asks the harder question,
  linking a ligand to the *downstream transcriptional response* it should induce in the receiver, so
  a prediction becomes falsifiable within the same dataset.
* [**COMMOT**](https://www.nature.com/articles/s41592-022-01728-4): treats signalling as optimal
  transport over the tissue, handling competition between receivers for a limited ligand supply.

### Where this Tutorial sits

Tutorial 1 read three platforms and put them on screen. Tutorial 2 asked which cell types sit
together and turned the answer into niches you could measure. This Tutorial took the same cells,
swapped the panel, and asked what might be passing between them, finding the same tumour-stroma
interfaces Tutorial 2's niches drew. **Tutorial 4** leaves transcripts behind entirely and asks what
could have been inferred from the H&E image alone.

The through-line: **every one of these methods has a boundary, and the boundary usually lands on the
immune compartment.** Tutorial 2 could not resolve a B cell cluster. This Tutorial ranks every immune
pair below the abundant housekeeping and adhesion biology, and cannot see PD-L1/PD-1 at all.